In [1]:
# Imports and Setup
import pandas as pd
import numpy as np
import mne
from glob import glob
from os.path import join as ospj
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ===== CONFIGURATION (Standalone) =====
# Data directories
DATA_DIR = "/mnt/leif/littlab/users/pattnaik/ieeg_sz_embedding/data"
RAW_DATA_DIR = "/mnt/leif/littlab/data/Human_Data/CNT_iEEG_BIDS"

# Load seizure times from Excel
sz_times = pd.read_excel(
    ospj(DATA_DIR, "metadata/Manual validation.xlsx"),
    sheet_name="AllSeizureTimes",
    index_col=0,
)

# Load RID-HUP mapping table
rid_hup_table = pd.read_csv(
    ospj(DATA_DIR, "metadata/rid_hup_table.csv"), index_col=0
)
rid_hup_table.dropna(inplace=True, subset=["hupsubjno"])
for ind, row in rid_hup_table.iterrows():
    rid_hup_table.loc[ind, "hupsubjno"] = int(row["hupsubjno"][:3])

rid_hup_table.index = [f"sub-RID{x:04d}" for x in rid_hup_table.index]
rid_hup_table['hupsubjno'] = [f"HUP{x:03d}" for x in rid_hup_table['hupsubjno']]

# Create mapping dictionaries
rid_to_hup = rid_hup_table['hupsubjno'].to_dict()
hup_to_rid = {v: k for k, v in rid_to_hup.items()}

# Clean up sz_times
sz_times.dropna(inplace=True, subset=["IEEGname"])
sz_times = sz_times[sz_times["IEEGname"] != "HUP203_phaseII"]  # No RID for this subject

# Create sz_table with RID indices
sz_table = sz_times.copy()
sz_table.index = sz_table.index.map(hup_to_rid)
sz_table.reset_index(inplace=True)

print(f"✓ Loaded {len(sz_table)} seizures from sz_times")
print(f"✓ Columns: {sz_table.columns.tolist()}")
print(f"✓ Configuration loaded successfully (standalone mode)")


✓ Loaded 1501 seizures from sz_times
✓ Columns: ['Patient', 'IEEGID', 'IEEGname', 'start', 'end', 'source', 'notes', 'Semiology']
✓ Configuration loaded successfully (standalone mode)


In [2]:
# Build Seizure Cache Dictionary
def build_seizure_cache():
    """
    Build a cache dictionary mapping seizure indices to EDF file paths and seizure times.
    
    Returns:
        dict: Cache dictionary with structure:
              {seizure_index: {'edf_path': str, 'onset': float, 'offset': float, 'patient': str}}
    """
    seizure_cache = {}
    missing_files = []
    
    print("Building seizure cache...")
    for ind, row in tqdm(sz_table.iterrows(), total=len(sz_table)):
        # Find EDF file using glob pattern (same as make_dataset.py lines 49-54)
        fname = glob(
            ospj(
                RAW_DATA_DIR,
                row.Patient,
                "ses-clinical01/ieeg",
                f"{row.Patient}_ses-clinical01_task-ictal{int(row.start)}_*.edf"
            )
        )
        
        if len(fname) == 0:
            missing_files.append([ind, row.Patient, "No file found"])
            continue
        if len(fname) > 1:
            missing_files.append([ind, row.Patient, "Multiple files found"])
            continue
        
        # Store seizure info in cache
        seizure_cache[ind] = {
            'edf_path': fname[0],
            'onset': row.start if 'start' in row else None,
            'offset': row.end if 'end' in row else None,
            'patient': row.Patient
        }
    
    print(f"✓ Cached {len(seizure_cache)} seizures")
    print(f"✗ Missing files for {len(missing_files)} seizures")
    
    if missing_files:
        print("\nMissing files summary:")
        for idx, patient, reason in missing_files[:5]:  # Show first 5
            print(f"  - Index {idx} ({patient}): {reason}")
        if len(missing_files) > 5:
            print(f"  ... and {len(missing_files) - 5} more")
    
    return seizure_cache, missing_files

# Build the cache
seizure_cache, missing_files = build_seizure_cache()
print(f"\nCache built with {len(seizure_cache)} seizures")


Building seizure cache...


100%|██████████| 1501/1501 [00:01<00:00, 911.67it/s] 

✓ Cached 1499 seizures
✗ Missing files for 2 seizures

Missing files summary:
  - Index 499 (sub-RID0365): No file found
  - Index 921 (sub-RID0472): No file found

Cache built with 1499 seizures


In [3]:
# Load Seizure Data Function
def load_seizure_data(seizure_idx, padding_before=30, padding_after=15):
    """
    Load and crop EDF file for a specific seizure with configurable time padding.
    
    Parameters:
    -----------
    seizure_idx : int
        Index of the seizure in the cache dictionary
    padding_before : float, default=30
        Time in seconds to include before seizure onset
    padding_after : float, default=15
        Time in seconds to include after seizure offset
    
    Returns:
    --------
    dict : Dictionary containing:
        - 'raw_cropped': mne.io.Raw object with the cropped data
        - 'seizure_idx': seizure index
        - 'patient': patient ID
        - 'onset': seizure onset time (from metadata)
        - 'offset': seizure offset time (from metadata)
        - 'seizure_start_in_file': time in file where seizure starts
        - 'seizure_duration': duration of seizure
        - 'tmin': start time of crop in file
        - 'tmax': end time of crop in file
        - 'padding_before': padding before onset
        - 'padding_after': padding after offset
    """
    
    # Check if seizure exists in cache
    if seizure_idx not in seizure_cache:
        raise ValueError(f"Seizure index {seizure_idx} not found in cache. "
                        f"Available indices: {list(seizure_cache.keys())[:10]}...")
    
    # Get seizure info from cache
    sz_info = seizure_cache[seizure_idx]
    edf_path = sz_info['edf_path']
    onset = sz_info['onset']
    offset = sz_info['offset']
    patient = sz_info['patient']
    
    print(f"Loading seizure {seizure_idx} for patient {patient}")
    print(f"  EDF file: {os.path.basename(edf_path)}")
    print(f"  Onset: {onset}s, Offset: {offset}s")
    
    # Load EDF file
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    
    # Get annotations to find where the seizure actually starts in the file
    # Based on make_dataset.py logic: if annotation onset is 30s, the seizure starts at 30s
    if len(raw.annotations) > 0 and raw.annotations.onset[0] == 30:
        seizure_start_in_file = 30.0
    else:
        seizure_start_in_file = 0.0
    
    # Calculate the seizure duration
    seizure_duration = offset - onset
    
    # Calculate time window in the file
    tmin = max(0, seizure_start_in_file - padding_before)
    tmax = min(raw.times[-1], seizure_start_in_file + seizure_duration + padding_after)
    
    print(f"  Cropping from {tmin:.1f}s to {tmax:.1f}s (duration: {tmax-tmin:.1f}s)")
    
    # Crop data to time window
    raw_cropped = raw.copy().crop(tmin=tmin, tmax=tmax)
    
    print(f"  Loaded {len(raw_cropped.ch_names)} channels at {raw_cropped.info['sfreq']} Hz")
    
    # Return all relevant information
    return {
        'raw_cropped': raw_cropped,
        'seizure_idx': seizure_idx,
        'patient': patient,
        'onset': onset,
        'offset': offset,
        'seizure_start_in_file': seizure_start_in_file,
        'seizure_duration': seizure_duration,
        'tmin': tmin,
        'tmax': tmax,
        'padding_before': padding_before,
        'padding_after': padding_after,
    }



In [4]:
load_seizure_data(0)

Loading seizure 0 for patient sub-RID0106
  EDF file: sub-RID0106_ses-clinical01_task-ictal8816_run-01_ieeg.edf
  Onset: 8816.75s, Offset: 8911.44s
  Cropping from 0.0s to 125.0s (duration: 125.0s)
  Loaded 125 channels at 500.0 Hz


{'raw_cropped': <RawEDF | sub-RID0106_ses-clinical01_task-ictal8816_run-01_ieeg.edf, 125 x 62500 (125.0 s), ~59.7 MB, data loaded>,
 'seizure_idx': 0,
 'patient': 'sub-RID0106',
 'onset': 8816.75,
 'offset': 8911.44,
 'seizure_start_in_file': 30.0,
 'seizure_duration': 94.69000000000051,
 'tmin': 0,
 'tmax': 124.998,
 'padding_before': 30,
 'padding_after': 15}